# Módulo 3 - IA Generativa

### 1. Introdução
**Contexto**

As autoridades querem relatórios claros e visuais para comunicar com decisores e cidadãos. Pretende-se usar IA Generativa para resumir resultados dos Módulos 1 e 2 e propor recomendações.

**Objetivos de Aprendizagem**
1.	Explorar engenharia de prompts para síntese textual;
2.	Criar conteúdos inovadores, mas responsáveis;
3.	Discutir limitações, riscos e implicações socioeconómicas.

**Tarefas**
1.	Desenvolver um script ou notebook gen_report.py que recebe dados em JSON/CSV e gera:
    1.	resumo executivo (≤ 200 palavras);
    2.	2-3 recomendações de ação;
    3.	secção "limitações e riscos".
2.	Testar ≥ 3 variantes de prompts e comentar diferenças.
3.	Incluir avaliação crítica: transparência, enviesamentos, possíveis "alucinações".


## Setup

Antes de correr este notebook, deve ter:

1. Um ficheiro `.env` na pasta `Module_3/` com `HF_TOKEN=hf_xxxxxxxxxxxxx` (token de https://huggingface.co/settings/tokens).
2. As bibliotecas instaladas:

```bash
pip install python-dotenv huggingface_hub pandas matplotlib
```

> **Nota:** As células de geração de relatório (secção 4) usam `gen_report.py` que suporta dois backends:
> - **Anthropic Claude** (se `ANTHROPIC_API_KEY` estiver definida no `.env`)
> - **Fallback offline** (sem chave — produz Markdown estático a partir dos factos)
>
> As células de experiência com temperatura (secção 5) usam o cliente HuggingFace diretamente, tal como os notebooks base.

In [ ]:
import sys
import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import dotenv_values
from huggingface_hub import InferenceClient

# Paths — notebook vive em Module_3/, root é o pai
ROOT         = Path("..").resolve()
ALERTS       = ROOT / "Module_1" / "resultados_alertas.csv"
RULES        = ROOT / "Module_1" / "regras.json"
METRICS      = ROOT / "Module_2" / "resultados" / "metrics.csv"
METRICS_COMP = ROOT / "Module_2" / "resultados" / "metrics_comparacao.csv"
DATA_CLEAN   = ROOT / "data" / "clean_air_quality.csv"

config = dotenv_values(".env")
token  = config.get("HF_TOKEN")
# print(token)
# Cliente HuggingFace — usado nas experiências de temperatura (secção 5)
hf_client = InferenceClient(model="Qwen/Qwen2.5-72B-Instruct", token=token)

def call_hf(prompt: str, temperature: float = 0.7, max_tokens: int = 800) -> str:
    r = hf_client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return r.choices[0].message["content"]

print("Setup concluído.")

## 2. Recolha de Factos dos Módulos Anteriores

**Princípio de grounding:** o LLM nunca vê o dataset bruto. Recebe apenas factos numéricos pré-calculados pelos Módulos 1 e 2. Isto restringe a sua função à formatação textual e mitiga o risco de alucinação — toda a decisão factual continua a vir de código determinístico.

A função `load_facts` em `gen_report.py` agrega os três ficheiros de input num único objeto `ReportFacts`.

In [ ]:
# Importar load_facts e os prompts do script gen_report.py
sys.path.insert(0, str(Path(".").resolve()))
from gen_report import load_facts, generate, PROMPT_BASELINE, PROMPT_ESTRUTURADO, PROMPT_CRITICO_ETICO

facts = load_facts(ALERTS, RULES, METRICS)
print(json.dumps(facts.to_dict(), ensure_ascii=False, indent=2))

## 3. Exploração dos Dados de Input

Antes de gerar texto, visualizamos o que o LLM vai receber — é boa prática auditar os factos antes de os injetar no prompt.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Gráfico 1: Distribuição de níveis de risco ---
level_order = ["NORMAL", "MODERADO", "ALTO"]
levels = facts.alerts_by_level
counts = [levels.get(l, 0) for l in level_order]
colors = ["#4caf50", "#ff9800", "#f44336"]
axes[0].bar(level_order, counts, color=colors)
axes[0].set_title("Alertas por Nível de Risco")
axes[0].set_ylabel("Nº de registos")
for i, v in enumerate(counts):
    axes[0].text(i, v + 5, str(v), ha="center", fontsize=9)

# --- Gráfico 2: % alertas por cidade ---
cities = list(facts.pct_alerts_by_city.keys())
pcts   = list(facts.pct_alerts_by_city.values())
axes[1].bar(cities, pcts, color=["#1976d2", "#42a5f5"])
axes[1].set_title("% Registos com Risco > NORMAL")
axes[1].set_ylabel("%")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
for i, v in enumerate(pcts):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)

# --- Gráfico 3: Top regras disparadas ---
top_rules = facts.top_rules[:5]
rule_ids  = [r["id"] for r in top_rules]
rule_cnts = [r["count"] for r in top_rules]
axes[2].barh(rule_ids[::-1], rule_cnts[::-1], color="#7b1fa2")
axes[2].set_title("Top Regras Disparadas")
axes[2].set_xlabel("Nº de ocorrências")

plt.tight_layout()
plt.show()

In [ ]:
# Tabela de métricas dos modelos
metrics_df = pd.read_csv(METRICS)

print("=== Classificação ===")
cls_cols = ["modelo", "accuracy", "precision", "recall", "f1", "roc_auc"]
display(metrics_df[metrics_df["tarefa"] == "classificacao"][cls_cols].reset_index(drop=True))

print("\n=== Regressão ===")
reg_cols = ["modelo", "r2", "mse", "mae"]
display(metrics_df[metrics_df["tarefa"] == "regressao"][reg_cols].reset_index(drop=True))

In [ ]:
# Comparação entre variantes do dataset (baseline / sem CO / SMOTE / sem CO + SMOTE)
comp_df = pd.read_csv(METRICS_COMP)
display(comp_df.sort_values(["modelo", "combo"]).reset_index(drop=True))

## 4. Engenharia de Prompts — 3 Variantes

O mesmo objeto `facts` é serializado em JSON e injetado em três prompts distintos. A qualidade e estrutura da saída dependem quase exclusivamente do prompt — os dados são idênticos.

| Variante | Audiência | Restrições explícitas | Tom |
|---|---|---|---|
| Baseline | Genérica | Nenhuma | Livre |
| Estruturado | Proteção Civil (técnico) | Cabeçalhos, limite de palavras, gatilhos quantitativos | Técnico |
| Crítico-Ético | Decisores + cidadãos | Grupos de risco, fontes regulamentares, transparência | Institucional/ético |

### 4.1 Variante 1 — Baseline

Prompt minimalista: pede os três blocos mas não especifica estrutura, tom, nem fonte das alegações. O modelo tem liberdade máxima — o que pode ser útil ou problemático.

In [ ]:
print("--- TEMPLATE DO PROMPT BASELINE ---")
print(PROMPT_BASELINE)
print()

In [ ]:
saida_baseline = generate(facts, "baseline")
print(saida_baseline)

### 4.2 Variante 2 — Estruturado

Adiciona regras explícitas: cabeçalhos Markdown obrigatórios, limite de 200 palavras no resumo, cada recomendação deve incluir o gatilho quantitativo (ex: `PM2.5 > 25 µg/m³`) e a fonte regulamentar. Proíbe expressamente inventar dados.

In [ ]:
print("--- TEMPLATE DO PROMPT ESTRUTURADO ---")
print(PROMPT_ESTRUTURADO)
print()

In [ ]:
saida_estruturado = generate(facts, "estruturado")
print(saida_estruturado)

### 4.3 Variante 3 — Crítico-Ético

Além das restrições estruturais, exige: (i) nomear grupos de risco específicos (asmáticos, idosos, crianças, trabalhadores ao ar livre); (ii) discutir o desbalanço de classes (~10 % registos 'má qualidade'); (iii) mencionar cobertura geográfica limitada; (iv) tratar os modelos como ferramentas de apoio, não como decisores autónomos.

In [ ]:
print("--- TEMPLATE DO PROMPT CRÍTICO-ÉTICO ---")
print(PROMPT_CRITICO_ETICO)
print()

In [ ]:
saida_critico = generate(facts, "critico_etico")
print(saida_critico)

## 5. Efeito da Temperatura

A temperatura controla a aleatoriedade da distribuição de probabilidade sobre tokens:

- **0.0–0.3** → respostas conservadoras, repetíveis, próximas do mais provável. Ideal para relatórios institucionais.
- **0.4–0.7** → ponto de equilíbrio para tarefas conversacionais.
- **0.8–1.2** → mais criatividade e variedade; útil para brainstorming mas perigoso em contexto factual.

Testamos aqui o efeito no mesmo prompt de síntese, usando o cliente HuggingFace diretamente (modelo Qwen2.5-72B).

In [ ]:
def comparar_temperaturas(prompt: str, temperaturas: list | None = None) -> None:
    if temperaturas is None:
        temperaturas = [0.1, 0.5, 1.0]
    for t in temperaturas:
        print(f"\n{'=' * 60}\nTemperatura: {t}\n{'=' * 60}")
        print(call_hf(prompt, temperature=t, max_tokens=300))


prompt_sintese = f"""Com base nos seguintes factos sobre qualidade do ar em Lisboa e Porto, \
escreve um parágrafo de resumo executivo (máximo 80 palavras) em português europeu.

Factos:
{json.dumps(facts.to_dict(), ensure_ascii=False)}
"""

comparar_temperaturas(prompt_sintese)

**Observação:** Para relatórios institucionais sobre qualidade do ar, a temperatura baixa (0.1–0.3) produz texto mais estável e factualmente alinhado. Temperatura alta pode introduzir formulações não suportadas pelos dados — o equivalente textual de uma alucinação subtil.

## 6. Comparação das Variantes

Analisamos automaticamente as três saídas em seis dimensões: comprimento, estrutura, rigor quantitativo, referência a limitações, grupos de risco explícitos e linguagem ética.

In [ ]:
def contar_palavras(texto: str) -> int:
    return len(texto.split())

def tem_padrao(texto: str, padrao: str) -> bool:
    return bool(re.search(padrao, texto, re.I | re.UNICODE))

variantes = [
    ("Baseline",      saida_baseline),
    ("Estruturado",   saida_estruturado),
    ("Crítico-Ético", saida_critico),
]

rows = []
for nome, texto in variantes:
    rows.append({
        "Variante":               nome,
        "Palavras":               contar_palavras(texto),
        "Cabeçalhos (##)":        tem_padrao(texto, r"^##"),
        "Gatilhos quantitativos": tem_padrao(texto, r"µg/m|mg/m|\d+\s*%"),
        "Menciona limitações":    tem_padrao(texto, r"limita|restri|cobertura"),
        "Grupos de risco":        tem_padrao(texto, r"asmát|idosos|criança|trabalhador"),
        "Fonte regulamentar":     tem_padrao(texto, r"2008/50|OMS|IPMA|WHO"),
        "Linguagem ética":        tem_padrao(texto, r"vi[eé]s|incertez|transparên|autónomo"),
    })

comp_df = pd.DataFrame(rows).set_index("Variante")
display(comp_df)

## 7. Avaliação Crítica

### 7.1 Grounding e risco de alucinação

O principal risco dos LLMs em contexto factual é a **alucinação**: o modelo gera texto plausível mas não suportado pelos dados. A célula abaixo demonstra o problema ao enviar o mesmo pedido *sem factos* — o modelo responde com números plausíveis mas inventados.


In [ ]:
prompt_sem_grounding = """Descreve a qualidade do ar em Lisboa e Porto em 2024, \
com dados numéricos sobre PM2.5, NO2 e alertas emitidos."""

print("=== SEM GROUNDING (risco de alucinação) ===")
print(call_hf(prompt_sem_grounding, temperature=0.3, max_tokens=300))

print("\n" + "=" * 60)
print("=== COM GROUNDING (factos pré-calculados) ===")
prompt_com_grounding = f"""Com base APENAS nos factos abaixo, descreve a qualidade do ar \
em Lisboa e Porto. Não uses dados que não constes nos factos fornecidos.

Factos:
{json.dumps(facts.to_dict(), ensure_ascii=False, indent=2)}
"""
print(call_hf(prompt_com_grounding, temperature=0.3, max_tokens=300))

### 7.2 Desbalanço de classes e viés

O modelo ML foi treinado com um dataset desequilibrado: a grande maioria dos registos corresponde a qualidade do ar "boa". Um modelo que preveja sempre "bom" tem precisão elevada mas recall baixo na classe minoritária — o que é inaceitável em saúde pública, onde falsos negativos têm consequências concretas.

In [ ]:
data = pd.read_csv(DATA_CLEAN, sep=";")
n_total = len(data)
n_bad   = (data["air_quality_good"] == 0).sum()
pct_bad = 100 * n_bad / n_total

print(f"Total de registos: {n_total}")
print(f"Registos com má qualidade (target=0): {n_bad} ({pct_bad:.1f}%)")
print(f"Registos com boa qualidade (target=1): {n_total - n_bad} ({100 - pct_bad:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Boa qualidade", "Má qualidade"], [n_total - n_bad, n_bad], color=["#4caf50", "#f44336"])
ax.set_title("Distribuição do Target (air_quality_good)")
ax.set_ylabel("Nº de registos")
for i, v in enumerate([n_total - n_bad, n_bad]):
    ax.text(i, v + 5, str(v), ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Recall do melhor classificador na classe minoritária vs. precisão global
metrics = pd.read_csv(METRICS)
cls = metrics[metrics["tarefa"] == "classificacao"].copy()
best = cls.loc[cls["f1"].idxmax()]

print(f"Melhor classificador (maior F1): {best['modelo']}")
print(f"  precisão global : {best['accuracy']:.1%}")
print(f"  Recall (má qualidade): {best['recall']:.1%}")
print(f"  F1              : {best['f1']:.3f}")
print(f"  ROC-AUC         : {best['roc_auc']:.3f}")
print()
print("Implicação: mesmo com precisão >90%, o modelo falha em identificar")
print(f"{100*(1-best['recall']):.0f}% dos episódios reais de má qualidade do ar.")

### 7.3 Limitações dos dados e dos modelos

| Dimensão | Limitação | Impacto |
|---|---|---|
| **Cobertura geográfica** | Apenas Lisboa e Porto | Não generalizável a cidades interiores ou costeiras |
| **Cobertura temporal** | ~2 meses de dados | Sem sazonalidade completa (verão/inverno) |
| **Definição do target** | `NO₂ ≥ 30 µg/m³ ∧ humidade ≥ 80 %` | Não alinhada com índices oficiais (IQAR) |
| **Desbalanço de classes** | ~10 % registos "má qualidade" | Recall baixo na classe de interesse |
| **Alucinação do LLM** | Modelo pode gerar texto plausível mas não suportado | Grounding obrigatório; validação humana antes de publicar |
| **Autonomia** | Modelos são ferramentas de apoio | Cada alerta deve ser validado pela Proteção Civil |

## 8. Integração: gen_report.py

O script `gen_report.py` encapsula todo o pipeline — recolha de factos, seleção de variante e chamada ao LLM — numa única linha de terminal. A função `generate()` foi chamada nas secções anteriores; aqui mostramos a interface de linha de comandos e o ficheiro Markdown produzido.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "gen_report.py", "--variant", "critico_etico", "--output", "report_demo.md"],
    capture_output=True,
    text=True,
    cwd=Path(".").resolve(),
)
print(result.stdout or result.stderr)

In [ ]:
# Mostrar o relatório gerado
report_path = Path("report_demo.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Ficheiro report_demo.md não encontrado — corra a célula anterior primeiro.")

---

## Conclusões

1. **Engenharia de prompt determina a qualidade da saída.** A variante baseline e a crítico-ética recebem os mesmos factos mas produzem textos com estrutura, rigor e transparência muito diferentes.

2. **Temperatura baixa (≤ 0.3) é essencial para texto institucional.** Temperatura alta introduz variabilidade indesejada e aumenta o risco de formulações não suportadas pelos dados.

3. **Grounding é a principal mitigação de alucinação.** O LLM não deve ter acesso ao dataset bruto; apenas a factos numéricos pré-calculados e verificados por código determinístico.

4. **O desbalanço de classes tem implicações diretas na comunicação pública.** Um recall de ~76 % significa que ~24 % dos episódios reais de má qualidade não seriam detetados — este número deve estar visível em qualquer relatório público.

5. **Os modelos são ferramentas de apoio, não decisores.** Qualquer alerta gerado pelo sistema deve ser validado pela Proteção Civil antes de ser comunicado ao público.